In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import cv2
from google.colab import files
from PIL import Image

# Load MNIST dataset
mnist = keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data (scaling between 0 and 1) and expand dimensions
x_train = np.expand_dims(x_train, axis=-1) / 255.0
x_test = np.expand_dims(x_test, axis=-1) / 255.0

# Reduce dataset size for faster training
x_train, y_train = x_train[:20000], y_train[:20000]
x_test, y_test = x_test[:5000], y_test[:5000]

# Data augmentation to improve generalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
datagen.fit(x_train)

# Build optimized CNN model
model = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),

    keras.layers.Conv2D(128, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.4),

    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])

# Compile model with adaptive learning rate
optimizer = keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train model with data augmentation
model.fit(datagen.flow(x_train, y_train, batch_size=64), epochs=5, validation_data=(x_test, y_test))

# Evaluate model
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Save model in recommended format
model.save("mnist_digit_model.keras")

# Function to predict digit from an image
def predict_digit(img):
    img = img.reshape(1, 28, 28, 1)  # Reshape for model input
    prediction = np.argmax(model.predict(img))
    return prediction

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 46ms/step - accuracy: 0.3526 - loss: 2.2193 - val_accuracy: 0.1142 - val_loss: 4.4415
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.7241 - loss: 0.8574 - val_accuracy: 0.9114 - val_loss: 0.2836
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.8197 - loss: 0.5729 - val_accuracy: 0.9388 - val_loss: 0.1838
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.8559 - loss: 0.4531 - val_accuracy: 0.9526 - val_loss: 0.1436
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.8750 - loss: 0.3859 - val_accuracy: 0.9532 - val_loss: 0.1393
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9541 - loss: 0.1289
Test Accuracy: 95.32%


In [ ]:
# Function to upload image and predict digit
def upload_and_predict():
    uploaded = files.upload()
    for filename in uploaded.keys():
        img = Image.open(filename).convert('L')  # Convert to grayscale
        img = img.resize((28, 28))  # Resize to 28x28 pixels
        img = np.array(img) / 255.0  # Normalize pixel values
        img = cv2.threshold(img, 0.5, 1, cv2.THRESH_BINARY)[1]  # Improve contrast
        img = np.expand_dims(img, axis=-1)  # Ensure shape (28, 28, 1)

        plt.imshow(img.squeeze(), cmap='gray')
        plt.title("Uploaded Image")
        plt.axis("off")
        plt.show()

        predicted_digit = predict_digit(img)
        print(f"🔢 Predicted digit: {predicted_digit}")

# Call function to upload and predict
upload_and_predict()